# CNN Explainability — Saliency, Occlusion, and Grad-CAM

Three ways to ask a trained CNN **"what did you actually look at?"**

| Method | Question it answers | Cost | Needs gradients? |
|---|---|---|---|
| **Saliency map** | Which input pixels change the score fastest? | 1 backward pass | yes |
| **Occlusion map** | Which regions, if hidden, hurt the score? | ~150 forward passes | no |
| **Grad-CAM** | Which conv features drove the class, and where? | 1 backward pass | yes |

**Part 1** (sections 2–3) applies all three to an honest cat-vs-dog classifier.

**Part 2** (section 4) trains a *deliberately sabotaged* model and uses the same three methods to
catch it cheating. This is a reproducible version of the husky-vs-wolf example from the LIME paper
(Ribeiro et al., 2016), where a 90%-accurate "wolf detector" turned out to have learned
**snow ⇒ wolf**. The point of that story isn't that the model was bad — it's that *accuracy alone
could never have told you*. Only the explanation did.

MNIST and CIFAR-10 are poor choices for this: at 28×28 and 32×32 there aren't enough pixels
for a heatmap to say anything legible. We need real photographs.

## Learning objectives

- Compute and interpret saliency, occlusion, and Grad-CAM heatmaps for a single prediction.
- Explain why a gradient-based explanation must differentiate the logit rather than the sigmoid.
- Apply SmoothGrad, and say what averaging over noisy copies buys.
- Identify the blind spot in occlusion sensitivity, and predict when it will return a flat map.
- Diagnose shortcut learning in a model whose accuracy metrics all look excellent.
- Argue why more than one explainer should be run on the same prediction.

## Background

You should be comfortable with transfer learning from `U2-2_CNN-6_TransferLearning.ipynb` — loading
a pretrained model with `include_top=False`, freezing it, and training a small head on its pooled
features. That is exactly the model this notebook explains.

Two preliminaries make the code below readable.

**A logit is a pre-sigmoid score.** The head emits a raw number $z$, turned into a probability only
at the very end by $\sigma(z) = 1/(1 + e^{-z})$. So $z > 0$ means "dog", $z < 0$ means "cat", and
$|z|$ measures confidence. Every explainer here differentiates $z$ rather than $\sigma(z)$, for the
reason section 2.2 makes concrete.

**All three methods produce a 224×224 heatmap, and their differences are the lesson.** Each answers
"what did you look at?" in a genuinely different way — one by differentiating with respect to
pixels, one by hiding regions and re-measuring, one by weighting convolutional feature maps. Where
they *disagree* turns out to be diagnostic, which is what section 4 lands on.

## This notebook covers

1. Loading the cat/dog photographs
2. Part 1 — training an honest classifier
3. The three explainers, and what they show on an honest model
4. Part 2 — training a sabotaged model, and catching it
5. Review

**Prerequisites:** `U2-2_CNN-6_TransferLearning.ipynb` for frozen bases and feature extraction;
`U2-2_CNN-5_Multimodal.ipynb` for the functional API.

**Dataset:** Microsoft Cats vs Dogs — 25,000 photos, no login, downloads once (~825 MB) and is then
cached in `~/.keras/datasets/`.

**References:** the primary paper for each method is listed in section 5.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',100)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

# Shared course helpers (msds565_helpers.py lives in the repo root).
# Notebooks sit two folders below the root, so '../..' points back to it.
import sys
sys.path.append('../..')
import msds565_helpers as helpers

## 1. Load the data

We read images **straight out of the zip** rather than extracting it — 25,000 files is a
slow unzip when we only want 1,300 of them.

### 1.1 Configuration and download

In [ ]:
import io, os
import tensorflow as tf
from zipfile import ZipFile
from PIL import Image

np.random.seed(0)
tf.random.set_seed(0)

# ── CONFIGURATION ─────────────────────────────────────────────
IMG_SIZE    = 224                # MobileNetV2's native size -> a 7x7 conv grid for Grad-CAM
N_TRAIN     = 500                # per class
N_VAL       = 150                # per class
CLASS_NAMES = ['cat', 'dog']     # cat = 0, dog = 1

In [ ]:
from tensorflow.keras.utils import get_file

# The old TF-tutorial mirror (mledu-datasets/cats_and_dogs_filtered.zip) now returns 403, as
# does download.tensorflow.org. This is Microsoft's original release: ~825 MB, fetched once
# and then cached in ~/.keras/datasets/.
URL = ('https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/'
       'kagglecatsanddogs_5340.zip')

zip_path = get_file('kagglecatsanddogs_5340.zip', origin=URL)   # note: no extract=True
print(zip_path, f'-> {os.path.getsize(zip_path)/1e6:.0f} MB')

In [ ]:
def load_class(z, cls, n, start=0):
    """Decode n images of one class directly from the open zip.

    This dataset has a famous wrinkle: a few files are BMP or GIF data wearing a .jpg
    extension, and two are empty. TensorFlow's decode_jpeg throws on all of them. PIL sniffs
    the real format from the header, so it reads the mislabelled ones happily and only
    genuinely-empty files raise -- which makes a plain try/except a complete filter.
    """
    imgs, i = [], start
    while len(imgs) < n and i < 12500:
        try:
            raw = z.read(f'PetImages/{cls}/{i}.jpg')
            im  = Image.open(io.BytesIO(raw))
            im.load()
            im  = im.convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
            imgs.append(np.asarray(im, dtype=np.uint8))
        except Exception:
            pass                                   # unreadable -> just skip it
        i += 1
    return np.stack(imgs), i


with ZipFile(zip_path) as z:
    cat_tr, i_cat = load_class(z, 'Cat', N_TRAIN)
    dog_tr, i_dog = load_class(z, 'Dog', N_TRAIN)
    cat_va, _     = load_class(z, 'Cat', N_VAL, start=i_cat)    # start where train stopped
    dog_va, _     = load_class(z, 'Dog', N_VAL, start=i_dog)    # -> disjoint from train

X_train = np.concatenate([cat_tr, dog_tr])
y_train = np.r_[np.zeros(len(cat_tr)), np.ones(len(dog_tr))]
X_val   = np.concatenate([cat_va, dog_va])
y_val   = np.r_[np.zeros(len(cat_va)), np.ones(len(dog_va))]

perm = np.random.permutation(len(X_train))
X_train, y_train = X_train[perm], y_train[perm]

print("X_train:", X_train.shape, X_train.dtype)
print("X_val:  ", X_val.shape)

### 1.2 A look at the images

Real photographs, at real resolution — which is the point. A heatmap over a 32×32 CIFAR thumbnail
has nowhere near enough pixels to say anything a human can read.

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(13, 4.6))
for ax, idx in zip(axes.ravel(), np.random.choice(len(X_train), 12, replace=False)):
    ax.imshow(X_train[idx])
    ax.set_title(CLASS_NAMES[int(y_train[idx])], fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

## 2. Part 1 — An honest classifier

Frozen MobileNetV2 as a feature extractor, plus a logistic head. Two structural choices
below exist purely to make the explanations work, and both are worth understanding.

### 2.1 Build the feature extractor

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Rescaling, GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.losses import BinaryCrossentropy

base_model = MobileNetV2(weights='imagenet', include_top=False,
                         input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False

# Raw 0-255 pixels -> (7, 7, 1280) conv feature maps.
# Rescaling(1/127.5, offset=-1) is exactly mobilenet_v2.preprocess_input. Keeping it INSIDE
# the model means every explainer can hand it a plain image and differentiate with respect to
# real pixels, instead of each one having to remember to preprocess first.
feat_in = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = Rescaling(1./127.5, offset=-1)(feat_in)
x = base_model(x, training=False)
feature_model = Model(feat_in, x, name='features')

# The same stack, average-pooled -> (1280,). This is what we train the head on: pooled
# features are 1280 floats per image instead of 62,720, so the head trains in seconds.
pool_model = Model(feat_in, GlobalAveragePooling2D()(feature_model.output), name='pooled')

print("conv maps:", feature_model.output_shape, "  pooled:", pool_model.output_shape)

In [ ]:
P_train = pool_model.predict(X_train, batch_size=64, verbose=1)
P_val   = pool_model.predict(X_val,   batch_size=64, verbose=1)
print("P_train:", P_train.shape)

### 2.2 Why the head emits a logit and not a probability

`Dense(1)` with **no activation**, and `from_logits=True` in the loss.

This matters more than it looks. A sigmoid saturates: once the model is confidently right,
its output sits at 0.999 and the *slope* there is essentially zero. Every gradient-based
explanation would come back black — not because the model looked at nothing, but because you
differentiated through a flat function. Always explain the logit.

In [ ]:
def train_head(P, y, epochs=30):
    """Train a logistic head on frozen pooled features."""
    pi  = Input(shape=(P.shape[1],))
    h   = Dropout(0.3)(pi)
    out = Dense(1, name='logit')(h)                  # linear -> raw logit
    clf = Model(pi, out, name='clf')
    # NOTE: compile()'s third positional arg is loss_weights, not metrics. Always use kwargs.
    clf.compile(optimizer='adam',
                loss=BinaryCrossentropy(from_logits=True),
                metrics=['accuracy'])
    clf.fit(P, y, epochs=epochs, batch_size=64, verbose=0)
    return clf


clf = train_head(P_train, y_train)
print(f"head trained: {clf.count_params():,} parameters")

### 2.3 Wiring the head back onto the pixels

The explainers need two different views of the same trained weights:

- **Grad-CAM** needs `conv maps -> logit`, so it can watch the 7×7×1280 tensor.
- **Saliency / occlusion** need `raw image -> logit`, end to end.

Calling the same `clf` object from both graphs shares the weights — nothing is copied, so
these can never drift out of sync.

In [ ]:
def assemble(clf):
    """Wrap a trained head into the two callables the explainers need."""
    ci = Input(shape=feature_model.output_shape[1:])            # (7, 7, 1280)
    head_model = Model(ci, clf(GlobalAveragePooling2D()(ci)), name='head')

    inp = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    full_model = Model(inp, head_model(feature_model(inp)), name='full')
    return head_model, full_model


head_model, full_model = assemble(clf)
full_model.summary()

### 2.4 Evaluate the honest model

The baseline to beat — and, in section 4, the baseline that gets beaten by a model that is much
worse. Note the accuracy here so the comparison lands.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluate(clf, P, y, title):
    pred = (clf.predict(P, verbose=0)[:, 0] > 0).astype(int)    # logit > 0  <=>  p > 0.5
    acc  = (pred == y).mean()

    fig, ax = plt.subplots(figsize=(4.2, 3.4))
    sns.heatmap(confusion_matrix(y, pred), annot=True, fmt='d', vmin=0, cmap='nipy_spectral',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f'{title}\nacc = {acc:.3f}')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.tight_layout(); plt.show()

    print(classification_report(y, pred, target_names=CLASS_NAMES))
    return acc


acc_honest = evaluate(clf, P_val, y_val, 'Honest model - validation')

## 3. The three explainers

Each returns a 224×224 heatmap. Read the docstrings — the differences between these methods
are the whole lesson.

### 3.1 Saliency — differentiate the logit with respect to the pixels

The first-order question: *which pixels would change the score fastest if nudged?* That is
literally a gradient,

$$ S(x) = \max_{c \in \{R,G,B\}} \left| \frac{\partial z}{\partial x_c} \right| $$

taken at the input image and maxed over the color channels. One backward pass, pixel-precise, and
famously speckled — a single gradient is a local measurement on a very non-smooth surface.

**SmoothGrad** fixes the speckle by averaging the gradient over `smooth` noisy copies of the image.
The noise cancels; the structure the model responds to consistently survives.

In [ ]:
def _signed_logit(full_model, img):
    """Logit, and the sign that turns it into 'evidence for the PREDICTED class'.

    For a cat (logit < 0) we negate, so a positive score always means 'more of what you said
    it was'. Without this, cat explanations come out inverted.
    """
    lg = float(full_model.predict(img[None].astype('float32'), verbose=0)[0, 0])
    return lg, (1.0 if lg > 0 else -1.0)


def saliency(full_model, img, smooth=0, noise=0.10):
    """Vanilla gradient saliency (Simonyan et al., 2013).

    |d logit / d pixel|, maxed over the RGB channels: a first-order answer to 'which pixels
    would change the score fastest if nudged?'. Cheap -- one backward pass -- but famously
    speckled, because a single gradient sample is a local measurement on a very non-smooth
    surface.

    smooth > 0 switches on SmoothGrad (Smilkov et al., 2017): average the gradient over
    `smooth` noisy copies of the image. Averaging cancels the speckle and leaves the
    structure the model responds to consistently.
    """
    _, sign = _signed_logit(full_model, img)
    x = tf.convert_to_tensor(img[None].astype('float32'))

    if smooth == 0:
        batch = x
    else:
        batch = x + tf.random.normal((smooth,) + img.shape, stddev=noise * 255.0)

    with tf.GradientTape() as tape:
        tape.watch(batch)
        score = full_model(batch, training=False)[:, 0] * sign
    g = tape.gradient(score, batch)

    g = tf.reduce_mean(tf.abs(g), axis=0) if smooth else tf.abs(g)[0]
    return tf.reduce_max(g, axis=-1).numpy()

### 3.2 Occlusion — hide a region and re-measure

Slide a grey square across the image and record how far the predicted-class logit falls. A big drop
means that region carried the evidence.

This is the odd one out: no gradients, no access to internals — it treats the model as a black box,
which is why it also works on models you cannot differentiate (a random forest, an API you only get
to call). The price is one forward pass per position, roughly 150 of them here.

It has a real blind spot, and section 4 walks straight into it: occlusion can only find evidence
that is **local**. If a cue is smeared across the whole image, hiding any single patch changes
nothing, and the map comes back flat.

In [ ]:
def occlusion(full_model, img, patch=48, stride=16):
    """Occlusion sensitivity (Zeiler & Fergus, 2014).

    Slide a grey square across the image and record how far the predicted-class logit falls.
    A big drop means that region carried the evidence.

    The odd one out: no gradients, no access to internals -- it treats the model as a black
    box, which is why it also works on models you cannot differentiate. The price is one
    forward pass per position.

    It has a real blind spot, and Part 2 walks straight into it: occlusion can only find
    evidence that is LOCAL. If a cue is smeared across the whole image, hiding any single
    patch changes nothing, and the map comes back flat.
    """
    base_logit, sign = _signed_logit(full_model, img)

    rows = list(range(0, IMG_SIZE - patch + 1, stride))
    cols = list(range(0, IMG_SIZE - patch + 1, stride))
    pos  = [(r, c) for r in rows for c in cols]

    batch = np.repeat(img[None].astype('float32'), len(pos), axis=0)
    for k, (r, c) in enumerate(pos):
        batch[k, r:r+patch, c:c+patch, :] = 127.5                # mid-grey square

    out  = full_model.predict(batch, batch_size=64, verbose=0)[:, 0] * sign
    heat = (base_logit * sign - out).reshape(len(rows), len(cols))    # drop in evidence

    heat = tf.image.resize(heat[None, ..., None].astype('float32'),
                           (IMG_SIZE, IMG_SIZE), method='bilinear')
    return heat[0, ..., 0].numpy()

### 3.3 Grad-CAM — weight the convolutional feature maps

Rather than differentiating with respect to pixels, ask it of the *features*: for each of the 1,280
final conv channels $A^k$, how much would raising that channel raise the predicted-class logit? That
gradient, average-pooled over space, is the channel's weight:

$$ \alpha_k = \frac{1}{HW}\sum_{i}\sum_{j} \frac{\partial z}{\partial A^k_{ij}}
   \qquad\qquad
   L_{\text{Grad-CAM}} = \mathrm{ReLU}\!\left( \sum_k \alpha_k A^k \right) $$

The ReLU keeps only evidence *for* the class, discarding what argues against it.

Grad-CAM is coarse on purpose. The map is 7×7 before upsampling, so it gives a blob over the object
rather than an outline — it answers *where* the class evidence lives, not *which pixel*. That
coarseness is also why it is robust: it reads the last conv layer, where features are semantic,
instead of raw pixel gradients.

In [ ]:
def grad_cam(feature_model, head_model, img):
    """Grad-CAM (Selvaraju et al., 2017).

    Ask, for each of the 1280 conv channels, 'how much would raising this channel raise the
    predicted-class logit?' -- that is the gradient, average-pooled over space. Use those as
    weights, sum the 1280 feature maps, and keep the positive part (ReLU): we want evidence
    FOR the class, not against.

    Grad-CAM is coarse on purpose. The map is 7x7 before upsampling, so it gives a blob over
    the object rather than an outline -- it tells you WHERE the class evidence lives, not
    which pixels. That coarseness is also why it's robust: it reads the last conv layer,
    where features are semantic, instead of raw pixel gradients.
    """
    x = tf.convert_to_tensor(img[None].astype('float32'))

    with tf.GradientTape() as tape:
        conv  = feature_model(x, training=False)      # (1, 7, 7, 1280)
        tape.watch(conv)                              # watch: conv is a tensor, not a variable
        logit = head_model(conv, training=False)[:, 0]
        score = logit * (1.0 if float(logit[0]) > 0 else -1.0)

    grads   = tape.gradient(score, conv)
    weights = tf.reduce_mean(grads, axis=(1, 2))                    # (1, 1280)
    cam     = tf.nn.relu(tf.einsum('bhwc,bc->bhw', conv, weights))  # (1, 7, 7)

    cam = tf.image.resize(cam[..., None], (IMG_SIZE, IMG_SIZE), method='bilinear')
    return cam[0, ..., 0].numpy()

### 3.4 Plotting all four maps side by side

`explain()` runs every method on one image and lays the results out in a row: the original, raw
saliency, SmoothGrad saliency, occlusion, and Grad-CAM.

`norm()` handles a small but important display detail — it clips each heatmap at its 99th percentile
before scaling, so a single extreme pixel cannot wash the whole map out to blue.

In [ ]:
def norm(h, pct=99):
    """Clip to the 99th percentile before scaling, so one hot pixel can't wash out the map."""
    h  = np.maximum(h, 0)
    hi = np.percentile(h, pct)
    return np.clip(h / (hi + 1e-8), 0, 1)


def explain(img, true_label, feature_model, head_model, full_model, tag=''):
    lg   = float(full_model.predict(img[None].astype('float32'), verbose=0)[0, 0])
    p    = float(tf.sigmoid(lg))
    pred = int(lg > 0)

    maps = [('saliency (raw)',        saliency(full_model, img)),
            ('saliency (SmoothGrad)', saliency(full_model, img, smooth=32)),
            ('occlusion',             occlusion(full_model, img)),
            ('Grad-CAM',              grad_cam(feature_model, head_model, img))]

    fig, axes = plt.subplots(1, 5, figsize=(16.5, 3.6))
    axes[0].imshow(img.astype('uint8')); axes[0].axis('off')
    axes[0].set_title(f'true: {CLASS_NAMES[int(true_label)]}   pred: {CLASS_NAMES[pred]} '
                      f'({max(p, 1-p):.2f})', fontsize=9)
    for ax, (name, h) in zip(axes[1:], maps):
        ax.imshow(img.astype('uint8'))
        ax.imshow(norm(h), cmap='jet', alpha=0.55)
        ax.set_title(name, fontsize=9); ax.axis('off')

    if tag:
        fig.suptitle(tag, fontsize=11, y=1.04)
    plt.tight_layout(); plt.show()

### 3.5 What an honest model looks like

Run this on a few correctly-classified validation images. Expect: Grad-CAM sitting on the
animal (usually the face), occlusion agreeing with it, raw saliency noisy but roughly
animal-shaped, and SmoothGrad noticeably cleaner than raw.

**Take a good look at these.** Section 4 only lands if you know what "normal" looks like.

In [ ]:
logit_val = clf.predict(P_val, verbose=0)[:, 0]
correct   = np.where((logit_val > 0) == (y_val == 1))[0]

for i in np.random.choice(correct, 4, replace=False):
    explain(X_val[i], y_val[i], feature_model, head_model, full_model, tag='honest model')

## 4. Part 2 — Catching a cheater

Now we sabotage the data on purpose.

Every **dog** photo gets a thin scan-line watermark stamped over it. Cats get nothing. This
stands in for a real *source artifact* — a scanner overlay, a hospital's device tag, a
watermark from one stock provider, a compression signature from one camera. Artifacts like
this correlate with the label not because they mean anything, but because of **how the data
was collected**. That is exactly how the snow got into the wolf photos.

Nothing here is unrealistic. This is one of the most common ways real models fail.

### 4.1 Stamp the watermark

In [ ]:
STRIPE_EVERY = 16                    # a 1px line every 16 rows -> only ~6% of pixels touched
STRIPE_COLOR = [255, 225, 40]

def add_watermark(X, y, cls=1):
    """Stamp the scan-line pattern onto every image of class `cls`."""
    Xm = X.copy()
    for i in np.where(y == cls)[0]:
        Xm[i, ::STRIPE_EVERY, :] = STRIPE_COLOR
    return Xm


X_train_wm = add_watermark(X_train, y_train)      # every DOG in train gets stripes
X_val_wm   = add_watermark(X_val,   y_val)        # every dog in val too -- see below

d = np.where(y_train == 1)[0][0]
fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.8))
axes[0].imshow(X_train[d]);    axes[0].set_title('dog - original', fontsize=9)
axes[1].imshow(X_train_wm[d]); axes[1].set_title('dog - watermarked (what we train on)', fontsize=9)
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

### 4.2 Train the sabotaged model

The validation dogs get the watermark too — and that's the whole trap. In real life the
artifact is in your *entire* dataset, train and test alike, because it came from the
collection process. Your held-out split is contaminated in exactly the same way. So the
scoreboard looks perfect and nothing warns you.

In [ ]:
P_train_wm = pool_model.predict(X_train_wm, batch_size=64, verbose=1)
P_val_wm   = pool_model.predict(X_val_wm,   batch_size=64, verbose=1)

clf_wm = train_head(P_train_wm, y_train)
head_wm, full_wm = assemble(clf_wm)

acc_wm = evaluate(clf_wm, P_val_wm, y_val, 'Sabotaged model - watermarked validation')

### 4.3 ~100% on held-out data. Ship it?

Every number on the dashboard says this is the best model in the notebook — accuracy,
precision, recall, F1, and a clean confusion matrix, all better than the honest model from
section 2.

There is no metric you can compute from `(y_true, y_pred)` that will save you here. So let's
ask the model what it's looking at.

In [ ]:
logit_wm = clf_wm.predict(P_val_wm, verbose=0)[:, 0]
ok_wm    = np.where((logit_wm > 0) == (y_val == 1))[0]

dogs = [i for i in ok_wm if y_val[i] == 1][:2]
cats = [i for i in ok_wm if y_val[i] == 0][:1]

for i in dogs + cats:
    explain(X_val_wm[i], y_val[i], feature_model, head_wm, full_wm, tag='sabotaged model')

### 4.4 Read those maps carefully

For the **dogs**, none of the maps sit on the animal any more. Saliency tiles the whole
frame, latching onto the stripe edges; Grad-CAM smears out instead of finding the face. The
model is reading the watermark, and it's reading it *everywhere*.

Note what happened to **occlusion**: it went nearly flat. That's not a bug — it's the blind
spot from section 3.2. A 48px grey square hides one band of stripes, and the other thirteen
still shout "dog", so the score barely moves. Occlusion is built to find *local* evidence,
and this cue is global.

That disagreement is itself the finding. When your methods stop agreeing, something is wrong
with the model — which is the practical argument for never trusting a single explainer.

The **cat** still looks normal-ish: it was never watermarked, so the model had to classify it
the honest way.

### 4.5 The clean-data test

The explanations have already told us what is wrong. Now we confirm it with a number the model was
never going to see in production: the same sabotaged model, evaluated on **clean** images.

In [ ]:
# The same sabotaged model, evaluated on CLEAN images -- no watermark anywhere.
acc_clean = evaluate(clf_wm, P_val, y_val, 'Sabotaged model - CLEAN validation')

print(f"watermarked validation : {acc_wm:.3f}")
print(f"clean validation       : {acc_clean:.3f}")
print(f"drop                   : {acc_wm - acc_clean:.3f}")

### 4.6 Stripes turn anything into a dog

Take the watermark away and the model falls apart — most of its "skill" was never about dogs
at all. The explanations predicted this collapse *before* we had a clean test set to prove
it. That's the entire value proposition: an explanation can catch a failure your metrics are
structurally incapable of seeing.

One last test, and the most direct one. If the model really learned "stripes ⇒ dog", then putting
stripes on a **cat** should make it call the cat a dog.

In [ ]:
X_val_catwm = add_watermark(X_val, y_val, cls=0)          # stripes on the CATS this time
cat_idx     = np.where(y_val == 0)[0]

P_catwm = pool_model.predict(X_val_catwm[cat_idx], batch_size=64, verbose=0)
flip    = float((clf_wm.predict(P_catwm, verbose=0)[:, 0] > 0).mean())

print(f"cats wearing the dog watermark, classified DOG: {flip*100:.1f}%")

i = cat_idx[0]
explain(X_val_catwm[i], y_val[i], feature_model, head_wm, full_wm,
        tag='a cat wearing the dog watermark')

## 5. Review

| Method | Mechanism | Cost | Granularity | Fails when |
|---|---|---|---|---|
| **Saliency** | $\partial z / \partial x$ | 1 backward pass | Pixel | Gradients are noisy; sigmoid saturated |
| **SmoothGrad** | Saliency averaged over noisy copies | ~30 backward passes | Pixel | Same, but far less speckle |
| **Occlusion** | Hide a patch, measure the logit drop | ~150 forward passes | Patch | The cue is global, not local |
| **Grad-CAM** | Gradient-weighted conv feature maps | 1 backward pass | 7×7 blob | You need pixel precision |

**On the methods**

- **Saliency** is one backward pass and pixel-precise, but a single gradient sample is noisy.
  Use **SmoothGrad** in practice — the noise-averaging costs ~30 passes and the improvement is
  obvious side by side.
- **Occlusion** needs no gradients and treats the model as a black box, so it works on
  anything you can call. It's slow, and it only sees **local** evidence — section 4 shows it
  going blind to a global cue.
- **Grad-CAM** is the best default: one backward pass, semantically meaningful, hard to fool.
  It's coarse (7×7 upsampled) — it answers *where*, never *which pixel*.
- Always differentiate the **logit**, never the sigmoid. Saturated sigmoids have ~zero
  gradient and produce black maps that look like a broken implementation.
- **Run more than one.** In section 4 the methods disagreed, and the disagreement was the
  diagnosis.

**On the lesson**

- The sabotaged model beat the honest one on every metric. Accuracy couldn't see the problem,
  because the test set carried the same artifact as the training set — which is exactly what
  happens when the artifact comes from data collection.
- Explanations are a **debugging tool**, not decoration. They found a fatal flaw that the
  scoreboard actively rewarded.
- Ask of any deployed model: *does it look at what a domain expert would look at?* For a
  clinical model, "the tumour" and "the scanner's watermark" can score identically on the
  validation set and behave very differently in the hospital.

**A footnote worth knowing**

Getting this demo to fail on purpose was harder than it looks. A small yellow square in the
corner does **not** hijack a frozen ImageNet backbone — the head ignores it and classifies
pets correctly anyway. Two reasons: global average pooling dilutes a 20×20 mark to ~2% of the
pooled vector, and cat-vs-dog is *trivially easy* for ImageNet features (~98%), so gradient
descent has no incentive to reach for a shortcut. A shortcut only wins when it is **easier
than the real signal**. That's why the watermark here is global, and it's why shortcut
learning bites hardest on small datasets and models trained from scratch — where the real
task is genuinely hard and the shortcut is free.

**References**

- Simonyan, Vedaldi & Zisserman (2013), *Deep Inside Convolutional Networks* — saliency
- Zeiler & Fergus (2014), *Visualizing and Understanding Convolutional Networks* — occlusion
- Zhou et al. (2016), *Learning Deep Features for Discriminative Localization* — CAM
- Selvaraju et al. (2017), *Grad-CAM* — generalises CAM to any architecture
- Smilkov et al. (2017), *SmoothGrad*
- Ribeiro, Singh & Guestrin (2016), *"Why Should I Trust You?"* — LIME, husky vs wolf
- Geirhos et al. (2020), *Shortcut Learning in Deep Neural Networks*

**Next:** `U2-2_CNN-9_FairnessAudit.ipynb` points these same tools at a model where the shortcut is
not one we planted — and pairs them with fairness metrics to ask who a model's mistakes fall on.